# 04 — Version 3: Collaborative Filtering & Version 4 Union Ranking

This notebook explores:
1. **Item-Based Collaborative Filtering (V3B)**: Primary model for movie-level queries.
2. **User-Based Collaborative Filtering (V3A)**: User-level neighborhood model for evaluation.
3. **Cold-Start Handling**: Insufficient rating threshold checks.
4. **Offline Evaluation**: Train/Test split (80/20), RMSE, MAE, Precision@K, Recall@K, Hit Rate@K, NDCG@K.
5. **Version 4 Union Ranker**: Ensemble layer fusing V1, V2, V3 with strict 7-tier priority hierarchy.


In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data_loader import load_all_data
from src.preprocessing import preprocess_movies, preprocess_ratings
from src.version3.item_based import ItemBasedCF
from src.version3.user_based import UserBasedCF
from src.version4.union_ranker import UnionRanker

users_df, movies_df, ratings_df = load_all_data()
movies_clean = preprocess_movies(movies_df)
ratings_clean = preprocess_ratings(ratings_df)

item_cf = ItemBasedCF(min_ratings=20)
item_cf.fit(movies_clean, ratings_clean)
print("Item-User Sparse Matrix shape:", item_cf.movie_user_sparse.shape)


## Item-Based CF Recommendations for Toy Story (MovieID = 1)


In [ ]:
recs_v3, cold_start_warning = item_cf.recommend(1, top_k=10)
print("Cold-start warning:", cold_start_warning)
recs_df = pd.DataFrame(recs_v3)
recs_df[['rank', 'title', 'genres', 'collaborative_score']]


## Version 4 Union Ranking Walkthrough


In [ ]:
from src.version1.basic_content import BasicContentRecommender
from src.version2.advanced_content import AdvancedContentRecommender

v1 = BasicContentRecommender().fit(movies_clean).recommend(1, top_k=10)
v2 = AdvancedContentRecommender().fit(movies_clean).recommend(1, top_k=10)
v3 = recs_v3

union_ranker = UnionRanker()
v4_recs = union_ranker.rank(v1, v2, v3, query_movie_id=1, top_k=10)

v4_df = pd.DataFrame(v4_recs)
v4_df[['rank', 'title', 'priority_group', 'recommended_by', 'model_agreement', 'combined_score']]
